# 방식 A: 모멘텀 종목 선정 + %R 타이밍 진입/청산

**구조:**
- 매월 초: 모멘텀(12-1 & 6-1) 랭킹으로 타깃 N종목 확정
- 신규 종목은 "대기 리스트"에 등록, %R(14) 매수 신호 대기
- %R(14)가 -80 이하 → -80 위로 크로스: **다음날 시가** 매수
- %R(14)가 -20 이상 → -20 아래로 크로스 + 쿨다운 만료: **다음날 시가** 매도
- 폴백: 월말까지 %R 매수 신호 없으면 월말 종가로 강제 매수

**룩어헤드 방지:**
- 모멘텀 신호: 전월 말 확정 종가 (월별 shift)
- %R(14): 당일까지 확정된 H/L/C로 계산, 신호 발생 → **다음날 시가** 체결
- 백테스트 루프: 일간

In [ ]:
# ============================================================
# [CELL 1] CONFIG
# ============================================================

CONFIG = {
    # ── 기간 ──
    "START_DATE": "2010-01-01",
    "END_DATE": "2025-01-01",
    "WARMUP_MONTHS": 13,

    # ── 모멘텀 ──
    "N_MOM": 8,
    "MOM_WEIGHT_12": 0.5,
    "MOM_WEIGHT_6": 0.5,

    # ── %R ──
    "WR_PERIOD": 14,
    "WR_BUY_LEVEL": -80,     # 이 아래에서 위로 크로스 → 매수
    "WR_SELL_LEVEL": -20,    # 이 위에서 아래로 크로스 → 매도

    # ── 쿨다운 ──
    "COOLDOWN_MONTHS": 3,

    # ── 비용 & 자본 ──
    "COST_RATE": 0.003,
    "INITIAL_CAPITAL": 100_000_000,

    # ── 폴백 ──
    "FORCE_BUY_ON_MONTH_END": True,  # 월말까지 %R 신호 없으면 강제 매수

    # ── 종목 ──
    "TICKER_CSV": "코스피 지수 198종목.csv",
    "TICKER_CSV_ENCODING": "cp949",
    "TICKER_COL_CANDIDATES": ["Ticker", "종목코드", "Code", "Symbol"],
    "FALLBACK_TICKERS": [
        "005930.KS", "000660.KS", "035420.KS", "005380.KS",
        "035720.KS", "051910.KS", "000270.KS", "068270.KS",
        "006400.KS", "003670.KS", "028260.KS", "105560.KS",
    ],
    "MIN_HISTORY_DAYS": 260,
}

print("✅ CONFIG 설정 완료")
for k, v in CONFIG.items():
    if not isinstance(v, list):
        print(f"   {k}: {v}")

In [ ]:
# ============================================================
# [CELL 2] 라이브러리
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("fivethirtyeight")

print("✅ 라이브러리 로드 완료")

In [ ]:
# ============================================================
# [CELL 3] 종목 티커 로드
# ============================================================

csv_path = CONFIG["TICKER_CSV"]
if os.path.exists(csv_path):
    try:
        df_raw = pd.read_csv(csv_path, encoding=CONFIG["TICKER_CSV_ENCODING"], dtype=str)
        col = next(c for c in CONFIG["TICKER_COL_CANDIDATES"] if c in df_raw.columns)
        tickers = [f"{t.strip().zfill(6)}.KS" for t in df_raw[col]]
        print(f"CSV 로드 성공: {len(tickers)}종목")
    except Exception as e:
        print(f"CSV 파싱 실패({e}). 폴백 티커 사용.")
        tickers = CONFIG["FALLBACK_TICKERS"]
else:
    print(f"'{csv_path}' 없음. 폴백 티커 사용.")
    tickers = CONFIG["FALLBACK_TICKERS"]

print(f"총 {len(tickers)}개 티커 준비")

In [ ]:
# ============================================================
# [CELL 4] 데이터 다운로드: 일간 OHLC + 월간 모멘텀
# ============================================================
#
# ★ 일간 데이터: Open(체결), High/Low/Close(%R 계산)  
# ★ 월간 데이터: 모멘텀 신호 (전월 말 확정 종가 기반)
# ★ %R 계산은 일간 H/L/C로 수행, shift 없이 당일 확정값 사용
#   → 신호 발생 다음날 시가로 체결하므로 룩어헤드 없음
#

stocks_daily = {}    # {ticker: DataFrame[Open, High, Low, Close, WR]}
stocks_monthly = {}  # {ticker: DataFrame[Close, Momentum]}

WR_N = CONFIG["WR_PERIOD"]

for t in tqdm(tickers, desc="데이터 다운로드",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{percentage:3.0f}%]"):
    try:
        df = yf.download(t, CONFIG["START_DATE"], CONFIG["END_DATE"],
                         progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        if len(df) < CONFIG["MIN_HISTORY_DAYS"]:
            continue

        # ── 일간: %R(14) 계산 ──
        highest = df["High"].rolling(WR_N).max()
        lowest  = df["Low"].rolling(WR_N).min()
        denom   = highest - lowest
        # %R = (Highest - Close) / (Highest - Lowest) * (-100)
        df["WR"] = np.where(denom > 0,
                            (highest - df["Close"]) / denom * (-100),
                            np.nan)

        stocks_daily[t] = df[["Open", "High", "Low", "Close", "WR"]].copy()

        # ── 월간: 모멘텀 계산 ──
        monthly = df["Close"].resample("ME").last().dropna().to_frame("Close")
        monthly["Mom_12_1"] = monthly["Close"].shift(1) / monthly["Close"].shift(12) - 1
        monthly["Mom_6_1"]  = monthly["Close"].shift(1) / monthly["Close"].shift(6) - 1
        w12, w6 = CONFIG["MOM_WEIGHT_12"], CONFIG["MOM_WEIGHT_6"]
        monthly["Momentum"] = w12 * monthly["Mom_12_1"] + w6 * monthly["Mom_6_1"]
        monthly = monthly.dropna(subset=["Momentum"])
        if len(monthly) > 0:
            stocks_monthly[t] = monthly

    except:
        continue

# 두 데이터 모두 있는 종목만 유지
valid_tickers = set(stocks_daily.keys()) & set(stocks_monthly.keys())
stocks_daily   = {t: v for t, v in stocks_daily.items() if t in valid_tickers}
stocks_monthly = {t: v for t, v in stocks_monthly.items() if t in valid_tickers}

print(f"\n분석 가능 종목: {len(valid_tickers)}개")

In [ ]:
# ============================================================
# [CELL 5] 거래일 인덱스 & 리밸런싱 날짜
# ============================================================

# 전체 거래일 합집합
all_dates = sorted(set().union(*(df.index.tolist() for df in stocks_daily.values())))
daily_index = pd.DatetimeIndex(all_dates)

# 매월 첫 거래일 = 리밸런싱(종목 선정) 시점
rebal_dates = daily_index.to_series().groupby(
    [daily_index.year, daily_index.month]
).first().values
rebal_dates = pd.DatetimeIndex(sorted(rebal_dates))

# 매월 마지막 거래일 (폴백 강제 매수용)
month_end_dates = daily_index.to_series().groupby(
    [daily_index.year, daily_index.month]
).last().values
month_end_set = set(pd.DatetimeIndex(sorted(month_end_dates)))

# 워밍업 제거
warmup_end = pd.Timestamp(CONFIG["START_DATE"]) + pd.DateOffset(months=CONFIG["WARMUP_MONTHS"])
trade_start = rebal_dates[rebal_dates >= warmup_end][0]
trading_days = daily_index[daily_index >= trade_start]
rebal_dates = rebal_dates[rebal_dates >= trade_start]
rebal_set = set(rebal_dates)

print(f"트레이딩 기간: {trading_days[0].strftime('%Y-%m-%d')} ~ {trading_days[-1].strftime('%Y-%m-%d')}")
print(f"총 거래일: {len(trading_days)}일, 리밸런싱: {len(rebal_dates)}회")

In [ ]:
# ============================================================
# [CELL 6] 백테스트 엔진 (일간 루프)
# ============================================================
#
# ★ 데이터 흐름:
#   [매월 첫 거래일] 전월 말 모멘텀으로 타깃 종목 확정
#     → 기존 보유(쿨다운 유지) + 신규 = 대기 리스트
#   [매일] 대기 종목: %R 매수 크로스 감시 → 다음날 시가 매수
#          보유 종목: 쿨다운 만료 시 %R 매도 크로스 감시 → 다음날 시가 매도
#   [월말] 대기 종목 중 미매수 → 폴백 강제 매수 (옵션)
#
# ★ 룩어헤드 방지:
#   - %R은 당일 종가까지로 계산 (확정), 체결은 다음날 시가
#   - 모멘텀은 전월 말 확정 데이터
#   - "다음날 시가"가 없는 경우 (마지막 거래일 등) 건너뜀
#

def get_momentum(ticker, rebal_date):
    """리밸런싱일 이전 가장 최근 월말 모멘텀 반환."""
    if ticker not in stocks_monthly:
        return None
    df = stocks_monthly[ticker]
    avail = df.loc[:rebal_date - pd.Timedelta(days=1)]
    if avail.empty:
        return None
    return avail["Momentum"].iloc[-1]


def run_backtest_A():
    cash = CONFIG["INITIAL_CAPITAL"]
    holdings = {}      # {ticker: {"shares": int, "buy_date": Timestamp}}
    pending_buy = {}   # {ticker: True}  대기 리스트 (매수 신호 대기 중)
    sell_tomorrow = {}  # {ticker: True}  매도 예약 (다음날 시가 매도)
    buy_tomorrow = {}   # {ticker: True}  매수 예약 (다음날 시가 매수)

    target_tickers = []  # 현재 월의 타깃 종목 리스트

    history = []
    trade_log = []
    total_traded = 0.0

    N = CONFIG["N_MOM"]
    COOLDOWN = CONFIG["COOLDOWN_MONTHS"]
    COST = CONFIG["COST_RATE"]
    BUY_LV = CONFIG["WR_BUY_LEVEL"]
    SELL_LV = CONFIG["WR_SELL_LEVEL"]

    prev_wr = {}  # {ticker: 전일 %R값}

    trading_days_list = list(trading_days)

    for i, today in enumerate(trading_days_list):

        # ============================================
        # (A) 예약 주문 체결 (전일 신호 → 오늘 시가 체결)
        # ============================================

        # 매수 체결
        for t in list(buy_tomorrow.keys()):
            if t in stocks_daily and today in stocks_daily[t].index:
                open_price = stocks_daily[t].loc[today, "Open"]
                if pd.notna(open_price) and open_price > 0:
                    # 동일 비중 배분: 현재 포트 가치 / 타깃 종목 수
                    port_val = cash
                    for ht, hi in holdings.items():
                        if ht in stocks_daily and today in stocks_daily[ht].index:
                            port_val += hi["shares"] * stocks_daily[ht].loc[today, "Open"]
                    n_target = max(len(target_tickers), 1)
                    alloc = port_val / n_target
                    shares = int(alloc // open_price)
                    cost = shares * open_price * (1 + COST)
                    if shares > 0 and cash >= cost:
                        cash -= cost
                        total_traded += shares * open_price
                        holdings[t] = {"shares": shares, "buy_date": today}
                        trade_log.append((today, t, "BUY", shares, open_price))
                        pending_buy.pop(t, None)
            buy_tomorrow.pop(t, None)

        # 매도 체결
        for t in list(sell_tomorrow.keys()):
            if t in holdings and t in stocks_daily and today in stocks_daily[t].index:
                open_price = stocks_daily[t].loc[today, "Open"]
                if pd.notna(open_price) and open_price > 0:
                    traded_val = holdings[t]["shares"] * open_price
                    cash += traded_val * (1 - COST)
                    total_traded += traded_val
                    trade_log.append((today, t, "SELL", holdings[t]["shares"], open_price))
                    del holdings[t]
            sell_tomorrow.pop(t, None)

        # ============================================
        # (B) 리밸런싱일: 모멘텀 랭킹 → 타깃 확정
        # ============================================
        if today in rebal_set:
            # 모멘텀 랭킹 (전월 말 확정)
            mom_scores = []
            for t in valid_tickers:
                mom = get_momentum(t, today)
                if mom is not None and t in stocks_daily and today in stocks_daily[t].index:
                    mom_scores.append((t, mom))
            mom_scores.sort(key=lambda x: x[1], reverse=True)

            # 쿨다운 보호
            cooldown_keep = []
            for t, info in holdings.items():
                months_held = ((today.year - info["buy_date"].year) * 12
                               + (today.month - info["buy_date"].month))
                if months_held < COOLDOWN:
                    cooldown_keep.append(t)

            # 타깃 구성: 쿨다운 유지 + 모멘텀 상위
            target_tickers = cooldown_keep.copy()
            for t, _ in mom_scores:
                if len(target_tickers) >= N:
                    break
                if t not in target_tickers:
                    target_tickers.append(t)

            # 타깃에서 빠진 보유 종목: 쿨다운 만료 시 즉시 매도 예약
            for t in list(holdings.keys()):
                if t not in target_tickers:
                    months_held = ((today.year - holdings[t]["buy_date"].year) * 12
                                   + (today.month - holdings[t]["buy_date"].month))
                    if months_held >= COOLDOWN:
                        sell_tomorrow[t] = True

            # 신규 진입 대기 리스트 갱신
            pending_buy = {}
            for t in target_tickers:
                if t not in holdings and t not in buy_tomorrow:
                    pending_buy[t] = True

        # ============================================
        # (C) 일간 %R 신호 감시
        # ============================================

        # 대기 종목: %R 매수 크로스 체크
        for t in list(pending_buy.keys()):
            if t not in stocks_daily or today not in stocks_daily[t].index:
                continue
            wr_today = stocks_daily[t].loc[today, "WR"]
            wr_prev = prev_wr.get(t, np.nan)

            if pd.notna(wr_today) and pd.notna(wr_prev):
                # 크로스: 전일 <= -80 이고 오늘 > -80
                if wr_prev <= BUY_LV and wr_today > BUY_LV:
                    buy_tomorrow[t] = True

        # 보유 종목: %R 매도 크로스 체크 (쿨다운 만료 종목만)
        for t, info in holdings.items():
            months_held = ((today.year - info["buy_date"].year) * 12
                           + (today.month - info["buy_date"].month))
            if months_held < COOLDOWN:
                continue  # 쿨다운 미만 → 매도 불가
            if t in sell_tomorrow:
                continue  # 이미 매도 예약됨

            if t not in stocks_daily or today not in stocks_daily[t].index:
                continue
            wr_today = stocks_daily[t].loc[today, "WR"]
            wr_prev = prev_wr.get(t, np.nan)

            if pd.notna(wr_today) and pd.notna(wr_prev):
                # 크로스: 전일 >= -20 이고 오늘 < -20
                if wr_prev >= SELL_LV and wr_today < SELL_LV:
                    sell_tomorrow[t] = True

        # ============================================
        # (D) 월말 폴백: 대기 종목 강제 매수
        # ============================================
        if CONFIG["FORCE_BUY_ON_MONTH_END"] and today in month_end_set:
            for t in list(pending_buy.keys()):
                if t not in buy_tomorrow:  # 아직 매수 신호 안 뜬 종목
                    buy_tomorrow[t] = True  # 다음날(익월 첫 거래일) 시가 매수

        # ============================================
        # (E) 전일 %R 업데이트 & 포트 기록
        # ============================================
        for t in valid_tickers:
            if t in stocks_daily and today in stocks_daily[t].index:
                wr_val = stocks_daily[t].loc[today, "WR"]
                if pd.notna(wr_val):
                    prev_wr[t] = wr_val

        # 포트폴리오 가치 기록 (종가 기준)
        port_val = cash
        for t, info in holdings.items():
            if t in stocks_daily and today in stocks_daily[t].index:
                port_val += info["shares"] * stocks_daily[t].loc[today, "Close"]
        history.append((today, port_val))

    # 결과
    perf_df = pd.DataFrame(history, columns=["Date", "Value"]).set_index("Date")
    trade_df = pd.DataFrame(trade_log, columns=["Date", "Ticker", "Action", "Shares", "Price"])
    return perf_df, trade_df, total_traded


print("✅ 백테스트 엔진 정의 완료")

In [ ]:
# ============================================================
# [CELL 7] 백테스트 실행
# ============================================================

perf_df, trade_df, total_traded = run_backtest_A()

print(f"백테스트 완료")
print(f"  시작 자산: {CONFIG['INITIAL_CAPITAL']:>20,} 원")
print(f"  최종 자산: {perf_df['Value'].iloc[-1]:>20,.0f} 원")
print(f"  총 매매 건수: {len(trade_df)}건")
print(f"\n매매 유형 분포:")
print(trade_df["Action"].value_counts().to_string())

In [ ]:
# ============================================================
# [CELL 8] 성과 분석
# ============================================================

def compute_metrics(df, initial_capital, traded_sum):
    # 월간 리턴으로 지표 계산
    monthly = df["Value"].resample("ME").last().dropna()
    returns = monthly.pct_change().dropna()
    years = max((df.index[-1] - df.index[0]).days / 365.25, 0.1)

    final = df["Value"].iloc[-1]
    cagr = ((final / initial_capital) ** (1 / years) - 1) * 100
    cum_ret = (final / initial_capital - 1) * 100

    running_max = df["Value"].cummax()
    mdd = (df["Value"] / running_max - 1).min() * 100

    sharpe = (returns.mean() * 12) / (returns.std() * np.sqrt(12)) if returns.std() > 0 else 0
    turnover = (traded_sum / 2) / df["Value"].mean() / years * 100

    return {
        "누적 수익률": f"{cum_ret:.2f}%",
        "CAGR": f"{cagr:.2f}%",
        "MDD": f"{mdd:.2f}%",
        "Sharpe Ratio": f"{sharpe:.2f}",
        "연간 회전율": f"{turnover:.1f}%",
        "최종 자산": f"{final:,.0f} 원",
    }


metrics = compute_metrics(perf_df, CONFIG["INITIAL_CAPITAL"], total_traded)
print(f"{'='*50}")
print(f"  [방식 A] 전략 성과 요약")
print(f"{'='*50}")
summary_df = pd.DataFrame(list(metrics.items()), columns=["지표", "값"])
display(summary_df.style.hide(axis="index").set_properties(**{"text-align": "left", "padding": "8px"}))

In [ ]:
# ============================================================
# [CELL 9] 연도별 성과
# ============================================================

def yearly_analysis(df, initial_capital):
    yearly = df["Value"].resample("YE").last()
    yearly_ret = yearly.pct_change()
    yearly_ret.iloc[0] = yearly.iloc[0] / initial_capital - 1
    yearly_mdd = df.groupby(df.index.year)["Value"].apply(
        lambda x: (x / x.cummax() - 1).min()
    )
    return pd.DataFrame({
        "연도": yearly_ret.index.year,
        "수익률(%)": (yearly_ret.values * 100).round(2),
        "MDD(%)": (yearly_mdd.values * 100).round(2),
    })

yearly_df = yearly_analysis(perf_df, CONFIG["INITIAL_CAPITAL"])
print(f"{'='*50}")
print(f"  [방식 A] 연도별 성과")
print(f"{'='*50}")
display(yearly_df.style.hide(axis="index")
        .format({"수익률(%)": "{:.2f}", "MDD(%)": "{:.2f}"})
        .set_properties(**{"text-align": "center", "padding": "8px"}))

In [ ]:
# ============================================================
# [CELL 10] 월간 세부 통계
# ============================================================

monthly_val = perf_df["Value"].resample("ME").last().dropna()
monthly_ret = monthly_val.pct_change().dropna()

win = monthly_ret[monthly_ret > 0]
lose = monthly_ret[monthly_ret < 0]

detail = {
    "월간 승률": f"{len(win)/len(monthly_ret)*100:.1f}%",
    "평균 상승월": f"{win.mean()*100:.2f}%" if len(win) > 0 else "N/A",
    "평균 하락월": f"{lose.mean()*100:.2f}%" if len(lose) > 0 else "N/A",
    "Profit Factor": f"{abs(win.sum()/lose.sum()):.2f}" if len(lose) > 0 else "N/A",
    "최고 월간": f"{monthly_ret.max()*100:.2f}%",
    "최저 월간": f"{monthly_ret.min()*100:.2f}%",
}

print(f"{'='*50}")
print(f"  [방식 A] 월간 세부 통계")
print(f"{'='*50}")
detail_df = pd.DataFrame(list(detail.items()), columns=["항목", "값"])
display(detail_df.style.hide(axis="index").set_properties(**{"text-align": "left", "padding": "8px"}))

In [ ]:
# ============================================================
# [CELL 11] 시각화
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(16, 12), gridspec_kw={"height_ratios": [3, 1]})

ax1 = axes[0]
ax1.plot(perf_df.index, perf_df["Value"], lw=2, color="#2196F3")
ax1.axhline(y=CONFIG["INITIAL_CAPITAL"], color="gray", ls="--", alpha=0.5)
ax1.set_yscale("log")
ax1.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:,.0f}"))
ax1.set_title("[방식 A] 모멘텀 + %R 타이밍 진입/청산", fontsize=16, fontweight="bold")
ax1.set_ylabel("자산 (로그 스케일)")
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
dd = (perf_df["Value"] / perf_df["Value"].cummax() - 1) * 100
ax2.fill_between(dd.index, dd.values, 0, color="#F44336", alpha=0.4)
ax2.set_ylabel("낙폭 (%)")
ax2.set_title("Drawdown", fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 룩어헤드 방지 체크리스트

| # | 항목 | 처리 |
|---|------|------|
| 1 | 모멘텀 신호 | 월말 리샘플링 + shift(1) → 전월 말 확정 종가만 사용 |
| 2 | %R 계산 | 당일까지의 H/L/C로 계산 (확정 데이터) |
| 3 | %R 크로스 판정 | 전일 %R vs 당일 %R 비교 (둘 다 확정) |
| 4 | 매수/매도 체결 | 신호 발생 **다음날 시가(Open)** → 신호와 체결 시점 분리 |
| 5 | 포트폴리오 평가 | 당일 종가 기준 (체결가와 무관) |
| 6 | 폴백 매수 | 월말 종가 시점 결정 → 익월 첫 거래일 시가 체결 |